# 🐾 Train your first walking policy
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isaac-sim/IsaacLab/blob/develop/notebooks/training.ipynb)

Give ANYmal-D a job: **follow a velocity command without falling over**.

🛠️ Build → 🧪 Check → 🏃 Train → 📈 Inspect → 🎬 Compare

You'll create your own task using a maintained locomotion environment. The policy sees **48 state values** and produces **12 joint actions**. No image observations needed.

![Reference: ANYmal-D learning to walk](https://download.isaacsim.omniverse.nvidia.com/isaaclab/images/rl_learning_progression_anymald.gif)

*Reference training progression. Your own checkpoint comparison appears below.*


## 1 · Power on
Choose **Runtime → Change runtime type → GPU** and run downward. Setup and the first simulation launch take longer because they download assets and compile kernels.


In [ ]:
# @title Prepare your robot playground
import contextlib
import html
import os
import re
import signal
import subprocess
import sys
import time
import uuid
from pathlib import Path

from IPython.display import HTML, Video, display

SESSION = Path("/content/isaaclab_notebook_logs")
SESSION.mkdir(parents=True, exist_ok=True)


def run(command, *, cwd=None, env=None, check=True, label="Working"):
    """Show progress, save full logs, and stop child processes when interrupted."""
    log = SESSION / f"{uuid.uuid4().hex}.log"
    status = display(HTML(""), display_id=True)
    started = time.monotonic()
    with log.open("w") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            env=env,
            stdout=stream,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        try:
            while process.poll() is None:
                with log.open("rb") as reader:
                    reader.seek(max(0, log.stat().st_size - 16000))
                    tail = reader.read().decode(errors="replace")
                iterations = re.findall(r"Learning iteration\s+(\d+)/(\d+)", tail)
                progress = '<progress style="width:100%"></progress>'
                if iterations:
                    current, total = map(int, iterations[-1])
                    progress = f'<progress value="{current + 1}" max="{total}" style="width:100%"></progress>'
                elapsed = int(time.monotonic() - started)
                status.update(
                    HTML(
                        f'<div style="padding:14px;border:1px solid #76b900;border-radius:12px">'
                        f"<b>{html.escape(label)}</b> · {elapsed // 60}m {elapsed % 60}s"
                        f"{progress}<small>First launch may compile kernels and download assets.</small></div>"
                    )
                )
                with contextlib.suppress(subprocess.TimeoutExpired):
                    process.wait(timeout=1)
        except BaseException:
            with contextlib.suppress(ProcessLookupError):
                os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            status.update(HTML(f"Stopped. Full log: <code>{html.escape(str(log))}</code>"))
            raise
    text = log.read_text(errors="replace")
    tail = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", text[-6000:])
    state = "✅" if process.returncode == 0 else "❌"
    status.update(
        HTML(
            f'<div style="padding:14px;border:1px solid #aaa;border-radius:12px">'
            f"<b>{state} {html.escape(label)}</b> · {int(time.monotonic() - started)}s"
            f'<details><summary>Show log</summary><pre style="white-space:pre-wrap">'
            f"{html.escape(tail)}</pre></details><small>Full log: {html.escape(str(log))}</small></div>"
        )
    )
    if check and process.returncode:
        raise RuntimeError(f"{label} failed. Expand Show log above. Full log: {log}")
    return subprocess.CompletedProcess(command, process.returncode)


def video_snapshot():
    """Include modification times so rerunning a cell refreshes overwritten clips."""
    return {p: p.stat().st_mtime_ns for p in ROOT.rglob("*.mp4")}


def show_video(before, title="Your rollout"):
    videos = [p for p in ROOT.rglob("*.mp4") if before.get(p) != p.stat().st_mtime_ns]
    if not videos:
        raise RuntimeError("No fresh video was recorded. Expand the playback log above.")
    path = max(videos, key=lambda p: p.stat().st_mtime_ns)
    display(HTML(f"<h3>{html.escape(title)}</h3>"))
    display(Video(str(path), embed=True, width=640, html_attributes="controls loop muted playsinline"))
    return path


ISAACLAB_REF = "develop"  # @param {type:"string"}
ROOT = Path("/content/IsaacLab-3")
PROJECT = Path("/content/colab_locomotion")
RUN_ENV = os.environ | {"PYTHONUNBUFFERED": "1"}
RUN_ENV.pop("DISPLAY", None)

gpu = (
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if __import__("shutil").which("nvidia-smi")
    else None
)
if gpu is None or gpu.returncode:
    raise RuntimeError("Choose Runtime → Change runtime type → GPU, then rerun this cell.")
display(HTML(f"<p><b>GPU ready</b> · {html.escape(gpu.stdout.strip())}</p>"))
run([sys.executable, "-m", "pip", "install", "-q", "uv"], label="Installing uv")
if not ROOT.exists():
    run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            ISAACLAB_REF,
            "https://github.com/isaac-sim/IsaacLab.git",
            str(ROOT),
        ],
        label="Downloading Isaac Lab",
    )
else:
    branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=ROOT, text=True).strip()
    current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
    requested = subprocess.run(["git", "rev-parse", ISAACLAB_REF], cwd=ROOT, capture_output=True, text=True)
    if branch != ISAACLAB_REF and (requested.returncode or requested.stdout.strip() != current_commit):
        raise RuntimeError(f"{ROOT} contains {branch}; set ISAACLAB_REF to that branch or use a new ROOT.")
run(["uv", "sync", "--inexact", "--extra", "video"], cwd=ROOT, env=RUN_ENV, label="Installing simulation dependencies")
revision = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
display(HTML(f"<p>✅ Ready · Isaac Lab revision <code>{revision}</code></p>"))

## 2 · Give the robot a job
We inherit ANYmal-D's flat-ground environment and customize commands, rewards, and episode length.

| Policy sees | Policy controls | Gets rewarded for |
|---|---|---|
| Body velocity, gravity, command, joints, previous actions | 12 joint-position targets | Following commands, staying upright, smooth motion |

Expand the next cell to see the complete task package. Edit `env_cfg.py` inside it to try your own variation.


In [ ]:
# @title Create your environment and PPO configuration
from textwrap import dedent

files = {
    "pyproject.toml": """
        [build-system]
        requires = ["uv_build>=0.12.6,<0.13"]
        build-backend = "uv_build"

        [project]
        name = "colab-locomotion"
        version = "0.1.0"
        description = "A tiny external Isaac Lab locomotion task."
        requires-python = ">=3.12,<3.13"
        dependencies = []

        [project.entry-points."isaaclab.tasks"]
        colab_locomotion = "colab_locomotion.tasks"

        [tool.uv.build-backend]
        module-name = "colab_locomotion"
    """,
    "src/colab_locomotion/__init__.py": '''
        # Copyright (c) 2026, The Isaac Lab Project Developers (https://github.com/isaac-sim/IsaacLab/blob/main/CONTRIBUTORS.md).
        # All rights reserved.
        #
        # SPDX-License-Identifier: BSD-3-Clause

        """Colab locomotion task package."""
    ''',
    "src/colab_locomotion/tasks/__init__.py": '''
        # Copyright (c) 2026, The Isaac Lab Project Developers (https://github.com/isaac-sim/IsaacLab/blob/main/CONTRIBUTORS.md).
        # All rights reserved.
        #
        # SPDX-License-Identifier: BSD-3-Clause

        """Register the Colab locomotion environment."""

        import gymnasium as gym

        gym.register(
            id="Isaac-Colab-Velocity-Flat-AnymalD",
            entry_point="isaaclab.envs:ManagerBasedRLEnv",
            disable_env_checker=True,
            kwargs={
                "env_cfg_entry_point": "colab_locomotion.tasks.env_cfg:ColabAnymalDEnvCfg",
                "rsl_rl_cfg_entry_point": "colab_locomotion.tasks.agent_cfg:ColabAnymalDPPORunnerCfg",
                "default_agent": "rsl_rl",
            },
        )
    ''',
    "src/colab_locomotion/tasks/env_cfg.py": '''
        # Copyright (c) 2026, The Isaac Lab Project Developers (https://github.com/isaac-sim/IsaacLab/blob/main/CONTRIBUTORS.md).
        # All rights reserved.
        #
        # SPDX-License-Identifier: BSD-3-Clause

        """State-based ANYmal-D locomotion environment for the Colab tutorial."""

        from isaaclab.utils import configclass
        from isaaclab_tasks.core.velocity.config.anymal_d.flat_env_cfg import AnymalDFlatEnvCfg


        @configclass
        class ColabAnymalDEnvCfg(AnymalDFlatEnvCfg):
            """ANYmal-D flat-terrain velocity tracking tuned for a compact lesson."""

            def __post_init__(self):
                super().__post_init__()

                self.scene.num_envs = 512
                self.episode_length_s = 12.0

                self.commands.base_velocity.heading_command = False
                ranges = self.commands.base_velocity.ranges
                ranges.lin_vel_x = (0.25, 1.25)
                ranges.lin_vel_y = (-0.25, 0.25)
                ranges.ang_vel_z = (-0.5, 0.5)

                self.rewards.track_lin_vel_xy_exp.weight = 2.0
                self.rewards.feet_air_time.weight = 0.75
    ''',
    "src/colab_locomotion/tasks/agent_cfg.py": '''
        # Copyright (c) 2026, The Isaac Lab Project Developers (https://github.com/isaac-sim/IsaacLab/blob/main/CONTRIBUTORS.md).
        # All rights reserved.
        #
        # SPDX-License-Identifier: BSD-3-Clause

        """RSL-RL configuration for the Colab locomotion task."""

        from isaaclab.utils import configclass
        from isaaclab_tasks.core.velocity.config.anymal_d.agents.rsl_rl_ppo_cfg import (
            AnymalDFlatPPORunnerCfg,
        )


        @configclass
        class ColabAnymalDPPORunnerCfg(AnymalDFlatPPORunnerCfg):
            """PPO defaults with a notebook-specific log directory."""

            def __post_init__(self):
                super().__post_init__()
                self.experiment_name = "colab_anymal_d_flat"
                self.save_interval = 10
    ''',
}

for relative_path, content in files.items():
    path = PROJECT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(dedent(content).lstrip())
    print("wrote", path.relative_to(PROJECT))

## 3 · Register your task
Install the little package so Isaac Lab can find it by name. Subsequent cells use `--no-sync` to preserve this editable installation.


In [ ]:
# @title Install your task
run(
    ["uv", "pip", "install", "--python", str(ROOT / ".venv/bin/python"), "--editable", str(PROJECT)],
    cwd=ROOT,
    check=True,
    label="Registering your task",
)
run(
    [
        "uv",
        "run",
        "--no-sync",
        "python",
        "-c",
        "import gymnasium as gym; import colab_locomotion.tasks; print(gym.spec('Isaac-Colab-Velocity-Flat-AnymalD'))",
    ],
    cwd=ROOT,
    env=RUN_ENV,
    check=True,
    label="Registering your task",
)

In [ ]:
# @title Your policy at a glance
TASK = "Isaac-Colab-Velocity-Flat-AnymalD"
display(
    HTML(
        '<div style="display:flex;gap:12px;flex-wrap:wrap">'
        '<span style="padding:16px;border:1px solid #76b900;border-radius:12px">📥 <b>48</b> state values</span>'
        '<span style="padding:16px;border:1px solid #76b900;border-radius:12px">🧠 <b>PPO</b> policy</span>'
        '<span style="padding:16px;border:1px solid #76b900;border-radius:12px">📤 <b>12</b> joint actions</span>'
        "</div>"
    )
)

## 4 · Check the environment 🧪
Run 32 robots for 64 steps with random actions. A successful check means the environment can reset and step; learning comes next.


In [ ]:
# @title Check reset and stepping
run(
    [
        "uv",
        "run",
        "--no-sync",
        "isaaclab",
        "random_agent",
        "--task",
        TASK,
        "--num_envs",
        "32",
        "--max_steps",
        "64",
        "--viz",
        "none",
        "physics=newton_mjwarp",
    ],
    cwd=ROOT,
    env=RUN_ENV,
    check=True,
    label="Checking reset and stepping",
)
print("✅ Reset and random-action stepping succeeded.")

## 5 · Start learning 🏃
**Quick** checks the workflow in 30 iterations. **Full** runs 300 iterations with more robots; gait quality still depends on training.

Watch the progress bar, then compare saved checkpoints below.


In [ ]:
# @title Choose a training run
TRAINING_RUN = "quick"  # @param ["quick", "full"]
if TRAINING_RUN == "quick":
    num_envs, max_iterations = 256, 30
else:
    num_envs, max_iterations = 1024, 300

experiment_root = ROOT / "logs/rsl_rl/colab_anymal_d_flat"
runs_before = set(experiment_root.glob("*"))
run(
    [
        "uv",
        "run",
        "--no-sync",
        "isaaclab",
        "train",
        "--rl_library",
        "rsl_rl",
        "--task",
        TASK,
        "--num_envs",
        str(num_envs),
        "--max_iterations",
        str(max_iterations),
        "--seed",
        "42",
        "--run_name",
        f"colab_{TRAINING_RUN}",
        "physics=newton_mjwarp",
    ],
    cwd=ROOT,
    env=RUN_ENV,
    check=True,
    label="Training your walking policy",
)

new_runs = [p for p in experiment_root.glob("*") if p.is_dir() and p not in runs_before]
if len(new_runs) != 1:
    raise RuntimeError("Could not identify this training run. Check the training log.")
RUN_DIR = new_runs[0]
print("✅ Saved:", RUN_DIR.name)

## 6 · Is it learning? 📈
Reward should trend upward, episodes should last longer, and velocity error should shrink. Judge several iterations together.


In [ ]:
# @title Plot this run's learning curves
import json

import matplotlib.pyplot as plt

# Read TensorBoard with Isaac Lab's Python, not the separate Colab kernel.
reader = """
import json, sys
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
events = EventAccumulator(sys.argv[1], size_guidance={"scalars": 0}).Reload()
tags = ["Train/mean_reward", "Train/mean_episode_length", "Metrics/base_velocity/error_vel_xy"]
print(json.dumps({tag: [(p.step, p.value) for p in events.Scalars(tag)]
                  for tag in tags if tag in events.Tags()["scalars"]}))
"""
result = subprocess.run(
    ["uv", "run", "--no-sync", "python", "-c", reader, str(RUN_DIR)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
curves = json.loads(result.stdout)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), layout="constrained")
plots = [
    ("Train/mean_reward", "Reward ↑", "episode return"),
    ("Train/mean_episode_length", "Stay upright ↑", "steps"),
    ("Metrics/base_velocity/error_vel_xy", "Follow commands ↓", "velocity error [m/s]"),
]
for ax, (tag, title, unit) in zip(axes, plots):
    points = curves.get(tag, [])
    if points:
        x, y = zip(*points)
        ax.plot(x, y, color="#568a00", linewidth=2)
        ax.fill_between(x, y, alpha=0.12, color="#76b900")
    else:
        ax.text(0.5, 0.5, "Not recorded yet", ha="center", transform=ax.transAxes)
    ax.set(title=title, xlabel="iteration", ylabel=unit)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.15)
plt.show()

## 7 · Watch the difference 🎬
Replay the first saved checkpoint and the final one from **this run**, with the same evaluation seed and command settings. The first checkpoint already includes one PPO update.

**Watch:** does the robot stay upright longer and follow commands more smoothly?


In [ ]:
# @title Replay early and final checkpoints
checkpoints = sorted(RUN_DIR.glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))
if not checkpoints:
    raise FileNotFoundError("No checkpoint found. Finish the training cell first.")

for label, checkpoint in [("Early steps", checkpoints[0]), ("After training", checkpoints[-1])]:
    before = video_snapshot()
    run(
        [
            "uv",
            "run",
            "--no-sync",
            "isaaclab",
            "play",
            "--rl_library",
            "rsl_rl",
            "--task",
            TASK,
            "--checkpoint",
            str(checkpoint),
            "--num_envs",
            "1",
            "--seed",
            "42",
            "--video",
            "--video_length",
            "300",
            "--viz",
            "newton_gl",
            "physics=newton_mjwarp",
        ],
        cwd=ROOT,
        env=RUN_ENV,
        label=f"Recording · {label}",
    )
    show_video(before, f"{label} · {checkpoint.stem}")

## Keep your work
Download the task code, checkpoints, logs, and videos before Colab disconnects.

**Next challenge:** change one command range or reward weight, then repeat the check → train → compare loop. Give the experiment a new run name so you can compare results.


In [ ]:
# @title Download your experiment
import shutil

from google.colab import files as colab_files

bundle = Path("/content") / f"locomotion-{RUN_DIR.name}"
bundle.mkdir(exist_ok=True)
shutil.copytree(PROJECT, bundle / "task", dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
shutil.copytree(RUN_DIR, bundle / "run", dirs_exist_ok=True)
(bundle / "isaaclab_revision.txt").write_text(
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True)
)
archive = shutil.make_archive(str(bundle), "zip", bundle)
colab_files.download(archive)